# 📝 중급 예시 — 메모장 만들기

지금까지 배운 위젯과 레이아웃을 종합하여 **메모장(Notepad)** 애플리케이션을 만듭니다.

| 기능 | 사용 기술 |
|------|-----------|
| 메뉴 바 | `Menu`, `add_cascade`, `add_command` |
| 텍스트 영역 | `Text`, `Scrollbar` |
| 파일 열기/저장 | `filedialog.askopenfilename`, `asksaveasfilename` |
| 상태 표시줄 | `Label` + `pack(side='bottom')` |
| 제목 업데이트 | `root.title()` |

> 💡 이 예시는 앞으로 배울 **파일 입출력, 이벤트 바인딩, 대화상자**의 핵심을 미리 체험해 봅니다.


---
## 1. 메모장 구조 설계

메모장은 크게 4가지 영역으로 구성됩니다.

```
┌─────────────────────────────┐
│ 메뉴 바 (File / Edit / Help) │
├─────────────────────────────┤
│                              │
│   텍스트 영역 (Text)          │   ← 스크롤바 연결
│                              │
├─────────────────────────────┤
│ 상태 표시줄 (줄/글자 수 등)   │
└─────────────────────────────┘
```

**구성 요소:**
- `Menu`: 상단 메뉴 (파일, 편집, 도움말)
- `Text` + `Scrollbar`: 본문 입력 영역
- `Label`: 하단 상태 표시줄
- `filedialog`: 파일 열기/저장 대화상자


---
## 2. 기본 창과 텍스트 영역 만들기

`Text` 위젯에 `Scrollbar`를 연결하는 것이 핵심입니다.
스크롤바 명령(`command`)과 텍스트 영역의 Y 스크롤(`yscrollcommand`)을 **서로 연결**해야 합니다.


In [ ]:
# 메모장 기본 뼈대 — Text + Scrollbar
import tkinter as tk

root = tk.Tk()
root.title('메모장')
root.geometry('600x400')

# 스크롤바 생성 (우측)
scrollbar = tk.Scrollbar(root)
scrollbar.pack(side='right', fill='y')

# 텍스트 영역 생성 (스크롤바 연결)
text = tk.Text(root, wrap='word', yscrollcommand=scrollbar.set,
               font=('Consolas', 11))
text.pack(side='left', expand=True, fill='both')

# 스크롤바를 텍스트 영역에 연결
scrollbar.config(command=text.yview)

root.mainloop()


---
## 3. 메뉴 바 추가하기

`Menu` 위젯으로 상단에 **File / Edit / Help** 메뉴를 만듭니다.

- `add_cascade`: 상위 메뉴 (File, Edit, Help)
- `add_command`: 하위 항목 (열기, 저장, 종료 ...)
- `add_separator`: 구분선


In [ ]:
# 메뉴 바 추가
import tkinter as tk

root = tk.Tk()
root.title('메모장')
root.geometry('600x400')

text = tk.Text(root, wrap='word', font=('Consolas', 11))
text.pack(expand=True, fill='both')

# 메뉴 바 생성
menubar = tk.Menu(root)
root.config(menu=menubar)

# File 메뉴
filemenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='File', menu=filemenu)
filemenu.add_command(label='새로 만들기', command=lambda: print('새 파일'))
filemenu.add_command(label='열기', command=lambda: print('열기'))
filemenu.add_separator()
filemenu.add_command(label='저장', command=lambda: print('저장'))
filemenu.add_command(label='다른 이름으로 저장', command=lambda: print('다른 이름 저장'))
filemenu.add_separator()
filemenu.add_command(label='종료', command=root.quit)

# Edit 메뉴
editmenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='Edit', menu=editmenu)
editmenu.add_command(label='실행 취소', command=text.edit_undo)
editmenu.add_command(label='다시 실행', command=text.edit_redo)
editmenu.add_separator()
editmenu.add_command(label='전체 선택', command=lambda: text.tag_add('sel', '1.0', 'end'))

# Help 메뉴
helpmenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='Help', menu=helpmenu)
helpmenu.add_command(label='정보', command=lambda: print('메모장 v1.0'))

root.mainloop()


---
## 4. 파일 열기 / 저장 — `filedialog`

`filedialog` 모듈로 운영체제의 파일 대화상자를 띄웁니다.

| 함수 | 용도 |
|------|------|
| `askopenfilename()` | 파일 열기 대화상자 → 경로 반환 |
| `asksaveasfilename()` | 다른 이름으로 저장 → 경로 반환 |

파일을 읽거나 쓸 때는 `with open(...)` 구문을 사용해 자동으로 닫힙니다.


In [ ]:
# 파일 열기 / 저장 함수 구현
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.title('메모장')
root.geometry('600x400')

text = tk.Text(root, wrap='word', font=('Consolas', 11))
text.pack(expand=True, fill='both')

current_file = {'path': None}  # 현재 열린 파일 경로 (딕셔너리로 감싸 클로저 공유)

def open_file():
    path = filedialog.askopenfilename(
        title='파일 열기',
        filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
    )
    if path:  # 취소를 누르면 빈 문자열 반환
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        text.delete('1.0', 'end')   # 기존 내용 삭제
        text.insert('1.0', content) # 새 내용 삽입
        current_file['path'] = path
        root.title(f'메모장 - {path}')

def save_file():
    path = current_file['path']
    if path is None:
        # 저장 경로가 없으면 다른 이름으로 저장 대화상자
        path = filedialog.asksaveasfilename(
            title='다른 이름으로 저장',
            defaultextension='.txt',
            filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
        )
    if path:
        with open(path, 'w', encoding='utf-8') as f:
            f.write(text.get('1.0', 'end-1c'))  # 끝의 자동 개행 제거
        current_file['path'] = path
        root.title(f'메모장 - {path}')

# 메뉴 바 연결
menubar = tk.Menu(root)
root.config(menu=menubar)
filemenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='File', menu=filemenu)
filemenu.add_command(label='열기', command=open_file)
filemenu.add_command(label='저장', command=save_file)
filemenu.add_separator()
filemenu.add_command(label='종료', command=root.quit)

root.mainloop()


---
## 5. 상태 표시줄 추가

하단에 `Label`을 배치하여 **글자 수**와 **현재 파일 경로**를 표시합니다.
`text` 위젯의 내용이 바뀔 때마다 상태를 갱신하려면 이벤트 바인딩(`<<Modified>>`)을 사용합니다.


In [ ]:
# 상태 표시줄 추가
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.title('메모장')
root.geometry('600x400')

# 텍스트 영역 + 스크롤바
scrollbar = tk.Scrollbar(root)
scrollbar.pack(side='right', fill='y')
text = tk.Text(root, wrap='word', yscrollcommand=scrollbar.set,
               font=('Consolas', 11))
text.pack(side='left', expand=True, fill='both')
scrollbar.config(command=text.yview)

# 상태 표시줄 (하단)
status = tk.Label(root, text='준비', anchor='w', bd=1, relief='sunken',
                  bg='#f0f0f0', font=('Arial', 9))
status.pack(side='bottom', fill='x')

current_file = {'path': None}

def update_status(event=None):
    content = text.get('1.0', 'end-1c')
    char_count = len(content)
    word_count = len(content.split())
    path = current_file['path'] or '새 파일'
    status.config(text=f'글자 수: {char_count}  |  단어 수: {word_count}  |  파일: {path}')

text.bind('<KeyRelease>', update_status)  # 키를 뗄 때마다 갱신

def open_file():
    path = filedialog.askopenfilename(
        title='파일 열기',
        filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
    )
    if path:
        with open(path, 'r', encoding='utf-8') as f:
            text.delete('1.0', 'end')
            text.insert('1.0', f.read())
        current_file['path'] = path
        root.title(f'메모장 - {path}')
        update_status()

def save_file():
    path = current_file['path'] or filedialog.asksaveasfilename(
        title='다른 이름으로 저장', defaultextension='.txt',
        filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
    )
    if path:
        with open(path, 'w', encoding='utf-8') as f:
            f.write(text.get('1.0', 'end-1c'))
        current_file['path'] = path
        root.title(f'메모장 - {path}')
        update_status()

menubar = tk.Menu(root)
root.config(menu=menubar)
fm = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='File', menu=fm)
fm.add_command(label='열기', command=open_file)
fm.add_command(label='저장', command=save_file)
fm.add_separator()
fm.add_command(label='종료', command=root.quit)

update_status()  # 초기 상태
root.mainloop()


---
## 6. 완성된 메모장 — 모든 기능 통합

지금까지의 기능을 모두 합친 **최종 메모장**입니다.
- File: 새로 만들기, 열기, 저장, 다른 이름으로 저장, 종료
- Edit: 실행 취소, 다시 실행, 전체 선택
- Help: 프로그램 정보
- 상태 표시줄: 글자 수 / 단어 수 / 파일 경로
- 제목 표시줄: 현재 파일 경로


In [ ]:
# ============================================================
#  완성된 메모장 (Notepad) — 모든 기능 통합
# ============================================================
import tkinter as tk
from tkinter import filedialog, messagebox

root = tk.Tk()
root.title('메모장 - 새 파일')
root.geometry('700x500')

# ------------------------------------------------------------
# 1) 텍스트 영역 + 스크롤바
# ------------------------------------------------------------
scrollbar = tk.Scrollbar(root)
scrollbar.pack(side='right', fill='y')

text = tk.Text(root, wrap='word', yscrollcommand=scrollbar.set,
               font=('Consolas', 11), undo=True)
text.pack(side='left', expand=True, fill='both')
scrollbar.config(command=text.yview)

# ------------------------------------------------------------
# 2) 상태 표시줄
# ------------------------------------------------------------
status = tk.Label(root, text='준비', anchor='w', bd=1, relief='sunken',
                  bg='#f0f0f0', font=('Arial', 9))
status.pack(side='bottom', fill='x')

# ------------------------------------------------------------
# 3) 현재 파일 경로 (상태로 보관)
# ------------------------------------------------------------
current_file = {'path': None}

def update_title():
    name = current_file['path'] or '새 파일'
    root.title(f'메모장 - {name}')

def update_status(event=None):
    content = text.get('1.0', 'end-1c')
    chars = len(content)
    words = len(content.split())
    path = current_file['path'] or '새 파일'
    status.config(text=f'  글자 수: {chars}   |   단어 수: {words}   |   파일: {path}')

# ------------------------------------------------------------
# 4) 파일 메뉴 기능
# ------------------------------------------------------------
def new_file():
    text.delete('1.0', 'end')
    current_file['path'] = None
    update_title()
    update_status()

def open_file():
    path = filedialog.askopenfilename(
        title='파일 열기',
        filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
    )
    if path:
        with open(path, 'r', encoding='utf-8') as f:
            text.delete('1.0', 'end')
            text.insert('1.0', f.read())
        current_file['path'] = path
        update_title()
        update_status()

def save_file():
    path = current_file['path']
    if path is None:
        path = filedialog.asksaveasfilename(
            title='다른 이름으로 저장', defaultextension='.txt',
            filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
        )
    if path:
        with open(path, 'w', encoding='utf-8') as f:
            f.write(text.get('1.0', 'end-1c'))
        current_file['path'] = path
        update_title()
        update_status()

def save_as_file():
    path = filedialog.asksaveasfilename(
        title='다른 이름으로 저장', defaultextension='.txt',
        filetypes=[('텍스트 파일', '*.txt'), ('모든 파일', '*.*')]
    )
    if path:
        with open(path, 'w', encoding='utf-8') as f:
            f.write(text.get('1.0', 'end-1c'))
        current_file['path'] = path
        update_title()
        update_status()

# ------------------------------------------------------------
# 5) 편집 메뉴 기능
# ------------------------------------------------------------
def select_all():
    text.tag_add('sel', '1.0', 'end')

# ------------------------------------------------------------
# 6) 도움말 메뉴 기능
# ------------------------------------------------------------
def show_about():
    messagebox.showinfo('정보', '메모장 v1.0\nTkinter 학습용 예제')

# ------------------------------------------------------------
# 7) 메뉴 바 구성
# ------------------------------------------------------------
menubar = tk.Menu(root)
root.config(menu=menubar)

# File 메뉴
filemenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='File', menu=filemenu)
filemenu.add_command(label='새로 만들기', command=new_file, accelerator='Ctrl+N')
filemenu.add_command(label='열기', command=open_file, accelerator='Ctrl+O')
filemenu.add_separator()
filemenu.add_command(label='저장', command=save_file, accelerator='Ctrl+S')
filemenu.add_command(label='다른 이름으로 저장', command=save_as_file, accelerator='Ctrl+Shift+S')
filemenu.add_separator()
filemenu.add_command(label='종료', command=root.quit)

# Edit 메뉴
editmenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='Edit', menu=editmenu)
editmenu.add_command(label='실행 취소', command=text.edit_undo, accelerator='Ctrl+Z')
editmenu.add_command(label='다시 실행', command=text.edit_redo, accelerator='Ctrl+Y')
editmenu.add_separator()
editmenu.add_command(label='전체 선택', command=select_all, accelerator='Ctrl+A')

# Help 메뉴
helpmenu = tk.Menu(menubar, tearoff=False)
menubar.add_cascade(label='Help', menu=helpmenu)
helpmenu.add_command(label='정보', command=show_about)

# ------------------------------------------------------------
# 8) 단축키 바인딩
# ------------------------------------------------------------
root.bind('<Control-n>', lambda e: new_file())
root.bind('<Control-o>', lambda e: open_file())
root.bind('<Control-s>', lambda e: save_file())
root.bind('<Control-Shift-S>', lambda e: save_as_file())
root.bind('<Control-a>', lambda e: select_all())

text.bind('<KeyRelease>', update_status)

# 초기 상태
update_title()
update_status()

root.mainloop()


---
## 🎯 정리

| 요소 | 핵심 API | 역할 |
|------|----------|------|
| 메뉴 바 | `Menu`, `add_cascade`, `add_command` | File / Edit / Help 계층 메뉴 |
| 텍스트 영역 | `Text`, `Scrollbar`, `yscrollcommand` | 본문 입력 + 스크롤 |
| 파일 대화상자 | `filedialog.askopenfilename`, `asksaveasfilename` | 파일 열기/저장 경로 선택 |
| 상태 표시줄 | `Label` + `pack(side='bottom')`, `<KeyRelease>` | 글자 수 / 단어 수 / 파일 경로 표시 |
| 제목 업데이트 | `root.title()` | 현재 파일 경로를 창 제목에 표시 |
| 단축키 | `root.bind('<Control-s>', ...)` | 키보드 단축키 연결 |

> 💡 **핵심 포인트**:
> - `Text`와 `Scrollbar`는 **서로의 명령을 연결**해야 동작합니다.
> - `filedialog`는 경로만 반환하므로, 실제 파일 읽기/쓰기는 `open()`으로 직접 처리합니다.
> - `text.get('1.0', 'end-1c')`에서 `end-1c`는 **끝의 자동 개행 문자**를 제거합니다.
> - `undo=True` 옵션으로 실행 취소/다시 실행을 쉽게 구현할 수 있습니다.


---
## 🎯 연습 문제

1. **글꼴 크기 조절 메뉴 추가**: Edit 메뉴에 '글꼴 크기' 하위 메뉴를 만들고, 10/12/14/16/20 중 선택하면 `Text` 위젯의 글꼴 크기가 변경되도록 하세요. (`text.config(font=(..., size))`)

2. **실시간 단어 수 표시**: 상태 표시줄에 글자 수뿐만 아니라 **단어 수**와 **줄 수**도 함께 표시되도록 `update_status()` 함수를 수정하세요. (`text.index('end-1c').split('.')` 로 줄 수 확인)

3. **찾기/바꾸기 기능 추가**: Edit 메뉴에 '찾기' 항목을 추가하고, 간단한 찾기 대화상자(`Toplevel`)를 띄워 입력한 단어를 본문에서 하이라이트(`tag_add`) 하세요. 선택적으로 '바꾸기' 기능도 추가해 보세요.
